In [21]:
import os
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import rasterio
from rasterio.mask import mask
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds
from shapely.geometry import box

In [22]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie obliczeniowe: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"Karta graficzna: {torch.cuda.get_device_name(0)}")


Używane urządzenie obliczeniowe: cuda
Karta graficzna: NVIDIA GeForce RTX 3050 Laptop GPU


In [24]:
# ==============================================================================
# 1. KONFIGURACJA KANAŁÓW
# ==============================================================================

UAV_BANDS = {
    'B450': 3,   # 442.7 nm (Blue)
    'B531': 7,   # 522.1 nm (Green 1)
    'B550': 8,   # 542.9 nm (Green 2)
    'B570': 9,   # 561.5 nm (Green 3)
    'B680': 15,  # 682.4 nm (Red)
    'B705': 16,  # 702.7 nm (Red-Edge)
    'B750': 18,  # 741.2 nm (NIR 1)
    'B800': 21   # 801.8 nm (NIR 2)
}

PS_BANDS = {
    'B2': 2,     # Blue (490 nm)
    'B3': 3,     # Green I (531 nm)
    'B4': 4,     # Green (565 nm)
    'B6': 6,     # Red (665 nm)
    'B7': 7,     # Red-Edge (705 nm)
    'B8': 8      # NIR (865 nm)
}

EPS = 1e-8

In [25]:
@torch.no_grad()
def compute_7_indices_gpu(uav_stack_np, ndvi_threshold=0.10):
    t = torch.from_numpy(uav_stack_np).to(DEVICE, dtype=torch.float32)

    # Główna maska na podstawie pasma NIR 800 nm
    valid_mask = (t[7] > 0) & torch.isfinite(t[7])

    sample_vals = t[7][valid_mask]
    if sample_vals.numel() > 0:
        max_val = torch.quantile(sample_vals[::min(100, sample_vals.numel())], 0.95).item()
        if max_val > 2.0:
            t = t / 10000.0 if max_val > 100.0 else t / 255.0

    b_450, b_531, b_550, b_570 = t[0], t[1], t[2], t[3]
    b_680, b_705, b_750, b_800 = t[4], t[5], t[6], t[7]

    # 1. NDVI (800, 680) + Maska Gleby
    ndvi = (b_800 - b_680) / (b_800 + b_680 + EPS)
    veg_mask = valid_mask & (ndvi > ndvi_threshold)

    # 2. EVI (800, 680, 450)
    evi = 2.5 * ((b_800 - b_680) / (b_800 + 6.0 * b_680 - 7.5 * b_450 + 1.0 + EPS))

    # 3. SAVI (800, 680, L=0.5)
    L = 0.5
    savi = ((b_800 - b_680) / (b_800 + b_680 + L + EPS)) * (1.0 + L)

    # 4. RENDVI (750, 705) - używa pierwszego NIR
    rendvi = (b_750 - b_705) / (b_750 + b_705 + EPS)
    
    # 5. PRI (531, 570)
    pri = (b_531 - b_570) / (b_531 + b_570 + EPS)

    # 6. NDWI / ARI (550, 700/705) - liczę ARI zgodnie z Twoim kodem, jeśli NDWI podmień wzór
    ari = (1.0 / (b_550 + EPS)) - (1.0 / (b_705 + EPS))

    # 7. CIre (800, 705)
    cire = (b_800 / (b_705 + EPS)) - 1.0

    indices_cpu = {
        'NDVI': ndvi.cpu().numpy(),
        'SAVI': savi.cpu().numpy(),
        'EVI': evi.cpu().numpy(),
        'RENDVI': rendvi.cpu().numpy(),
        'CIre': cire.cpu().numpy(),
        'ARI': ari.cpu().numpy(),
        'PRI': pri.cpu().numpy()
    }
    
    valid_mask_cpu = valid_mask.cpu().numpy()
    veg_mask_cpu = veg_mask.cpu().numpy()

    del t, b_450, b_531, b_550, b_570, b_680, b_705, b_750, b_800
    del ndvi, savi, evi, rendvi, cire, ari, pri, valid_mask, veg_mask
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return indices_cpu, valid_mask_cpu, veg_mask_cpu

In [26]:
def process_pair_with_geojson(
    uav_tif_path,
    ps_tif_path,
    geojson_path,
    field_id="01W",
    flight_date="2026-10-01",
    ps_date="2026-10-02",
    ndvi_soil_threshold=0.10,
    fvc_min_threshold=0.05
):
    t0 = time.time()
    records = []

    # 1. Wczytanie poligonu pola z pliku GeoJSON
    gdf = gpd.read_file(geojson_path)
    if gdf.empty:
        raise ValueError(f"Plik GeoJSON jest pusty: {geojson_path}")

    with rasterio.open(ps_tif_path) as src_ps, rasterio.open(uav_tif_path) as src_uav:
        ps_crs, uav_crs = src_ps.crs, src_uav.crs

        # Przeprojektowanie poligonu GeoJSON do układu współrzędnych PlanetScope oraz UAV
        gdf_ps = gdf.to_crs(ps_crs)
        gdf_uav = gdf.to_crs(uav_crs)
        
        # Zabezpieczenie przed DeprecationWarning dla starszych i nowszych wersji geopandas
        field_geom_ps = gdf_ps.union_all() if hasattr(gdf_ps, 'union_all') else gdf_ps.unary_union

        # 2. Docięcie (clip) sceny PlanetScope dokładnie do granic GeoJSON-a
        ps_cropped, ps_transform = mask(src_ps, gdf_ps.geometry, crop=True, filled=True, nodata=0)
        ps_indices, ps_nir_valid = compute_ps_indices_cpu(ps_cropped)

        # 3. Docięcie (clip) 8 zaktualizowanych kanałów UAV do granic GeoJSON-a i wrzucenie na GPU
        uav_band_list = [
            UAV_BANDS['B450'], UAV_BANDS['B531'], UAV_BANDS['B550'],
            UAV_BANDS['B570'], UAV_BANDS['B680'], UAV_BANDS['B705'],
            UAV_BANDS['B750'], UAV_BANDS['B800']
        ]
        print(f" -> Docinanie UAV do GeoJSON ({field_id}) i liczenie wskaźników na GPU...")
        uav_cropped, uav_transform = mask(
            src_uav, gdf_uav.geometry, indexes=uav_band_list, crop=True, filled=True, nodata=0
        )

        uav_ind, valid_uav_mask, veg_uav_mask = compute_7_indices_gpu(
            uav_cropped, ndvi_threshold=ndvi_soil_threshold
        )
        del uav_cropped

        uav_h, uav_w = valid_uav_mask.shape
        rows, cols = ps_nir_valid.shape

        # 4. Upscaling do oczek 3x3 m z dociętej siatki PlanetScope
        for r in range(rows):
            for c in range(cols):
                # Pomijamy piksele satelity poza poligonem (NoData)
                if ps_nir_valid[r, c] <= 0 or np.isnan(ps_nir_valid[r, c]):
                    continue

                # Współrzędne oczka 3x3 m w układzie PlanetScope
                x_min, y_max = ps_transform * (c, r)
                x_max, y_min = ps_transform * (c + 1, r + 1)

                # Sprawdzenie, czy środek piksela 3x3 m faktycznie leży wewnątrz poligonu GeoJSON
                pixel_box = box(min(x_min, x_max), min(y_min, y_max), max(x_min, x_max), max(y_min, y_max))
                if not field_geom_ps.intersects(pixel_box.centroid):
                    continue

                # Przeliczenie granic oczka 3x3 m na układ drona (jeśli mają różne CRS)
                if ps_crs != uav_crs:
                    uav_xmin, uav_ymin, uav_xmax, uav_ymax = transform_bounds(
                        ps_crs, uav_crs, min(x_min, x_max), min(y_min, y_max), max(x_min, x_max), max(y_min, y_max)
                    )
                else:
                    uav_xmin, uav_ymin = min(x_min, x_max), min(y_min, y_max)
                    uav_xmax, uav_ymax = max(x_min, x_max), max(y_min, y_max)

                # Wycięcie odpowiedniego okna z wyliczonych macierzy UAV
                win = from_bounds(uav_xmin, uav_ymin, uav_xmax, uav_ymax, transform=uav_transform)
                r0 = max(0, int(round(win.row_off)))
                r1 = min(uav_h, int(round(win.row_off + win.height)))
                c0 = max(0, int(round(win.col_off)))
                c1 = min(uav_w, int(round(win.col_off + win.width)))

                if r1 <= r0 or c1 <= c0:
                    continue

                patch_valid = valid_uav_mask[r0:r1, c0:c1]
                total_valid = np.count_nonzero(patch_valid)
                expected_area = (r1 - r0) * (c1 - c0)

                if total_valid < 0.5 * expected_area or total_valid == 0:
                    continue

                patch_veg = veg_uav_mask[r0:r1, c0:c1]
                veg_count = np.count_nonzero(patch_veg)

                # Obliczenie FVC i odfiltrowanie oczek wg progu
                fvc = veg_count / float(total_valid)
                if fvc <= fvc_min_threshold:
                    continue

                row_dict = {
                    'field_id': field_id,
                    'flight_date': flight_date,
                    'ps_date': ps_date,
                    'row': r,
                    'col': c,
                    'x_center': (x_min + x_max) / 2.0,
                    'y_center': (y_min + y_max) / 2.0,
                    'FVC': round(fvc, 4),
                    'n_uav_veg_pixels': veg_count
                }

                # Pobranie uśrednionych wartości z drona
                for idx_name, idx_full_arr in uav_ind.items():
                    vals = idx_full_arr[r0:r1, c0:c1][patch_veg]
                    vals = vals[np.isfinite(vals)]
                    row_dict[f'UAV_{idx_name}'] = float(np.mean(vals)) if vals.size > 0 else np.nan

                # Pobranie odpowiadających wskaźników z satelity
                for idx_name, ps_arr in ps_indices.items():
                    row_dict[f'PS_{idx_name}'] = float(ps_arr[r, c])

                records.append(row_dict)

    print(f" -> Gotowe w {time.time() - t0:.2f} s | Wyciągnięto {len(records)} oczek 3x3 m.")
    return pd.DataFrame(records).dropna()

In [27]:
# ==============================================================================
# 4. WPISANIE WSZYSTKICH NALOTÓW I URUCHOMIENIE PĘTLI
# ==============================================================================

# Tutaj dopisujesz kolejne słowniki dla każdego nalotu i pola (01W oraz 02W)
# Pamiętaj, aby dodać klucz 'geojson_path' z właściwą ścieżką do każdego nalotu.
kampania = [
    # --- 1. NALOT: 10.08 ---
    {
        "field_id": "01W",
        "flight_date": "2026-08-10",
        "ps_date": "2026-08-09",
        "uav_tif": "../Wegrzce_tify/01W1008.tif",
        "ps_tif":  "../results/planet/planet_2026-08-09.tif",
        "geojson_path": "../results/geojson/wegrzce_aoi-10-08.geojson" # <-- Dodano ścieżkę
    },
    # --- 2. NALOT: 02.09 ---
    {
        "field_id": "01W",
        "flight_date": "2026-09-02",
        "ps_date": "2026-09-02",
        "uav_tif": "../Wegrzce_tify/01W0209.tif",
        "ps_tif":  "../results/planet/planet_2026-09-02.tif",
        "geojson_path": "../results/geojson/wegrzce_aoi-02-09.geojson" # <-- Dodano ścieżkę
    },
    # --- 3. NALOT: 09.09 ---
    {
        "field_id": "01W",
        "flight_date": "2026-09-09",
        "ps_date": "2026-09-09",
        "uav_tif": "../Wegrzce_tify/01W0909.tif",
        "ps_tif":  "../results/planet/planet_2026-09-09.tif",
        "geojson_path": "../results/geojson/wegrzce_aoi-09-09.geojson" # <-- Dodano ścieżkę
    },
    # --- 4. NALOT: 15.09 ---   
    {
        "field_id": "01W",
        "flight_date": "2026-09-15",
        "ps_date": "2026-09-16",
        "uav_tif": "../Wegrzce_tify/01W1509.tif",
        "ps_tif":  "../results/planet/planet_2026-09-16.tif",
        "geojson_path": "../results/geojson/wegrzce_aoi-15-09.geojson" # <-- Dodano ścieżkę
    },
    # --- 5. NALOT: 01.10 ---
    {
        "field_id": "01W",
        "flight_date": "2026-10-01",
        "ps_date": "2026-10-02",
        "uav_tif": "../Wegrzce_tify/01W0110.tif",
        "ps_tif":  "../results/planet/planet_2026-10-02.tif",
        "geojson_path": "../results/geojson/wegrzce_aoi-01-10.geojson" # <-- Dodano ścieżkę
    },
    # --- 6. NALOT: 03.10 ---
    {
        "field_id": "01W",
        "flight_date": "2026-10-03",
        "ps_date": "2026-10-04",
        "uav_tif": "../Wegrzce_tify/01W0310.tif",
        "ps_tif":  "../results/planet/planet_2026-10-04.tif",
        "geojson_path": "../results/geojson/wegrzce_aoi-03-10.geojson" # <-- Dodano ścieżkę
    }
]

# Pętla przechodząca po wszystkich wpisanych wyżej nalotach:
all_dfs = []

for item in kampania:
    # Upewnij się, że sprawdzamy istnienie pliku GeoJSON w warunku
    if os.path.exists(item["uav_tif"]) and os.path.exists(item["ps_tif"]) and os.path.exists(item["geojson_path"]):
        print(f"\nPrzetwarzanie: Pole {item['field_id']} | Dron: {item['flight_date']} vs Satelita: {item['ps_date']}")
        # Zakładam, że Twoja funkcja to teraz `process_pair_with_geojson`, jak w poprzedniej poprawionej wersji
        df_pair = process_pair_with_geojson(
            uav_tif_path=item["uav_tif"],
            ps_tif_path=item["ps_tif"],
            geojson_path=item["geojson_path"], # Przekazujemy nowo dodaną ścieżkę
            field_id=item["field_id"],
            flight_date=item["flight_date"],
            ps_date=item["ps_date"],
            ndvi_soil_threshold=0.10,  # Maska gleby NDVI > 0.1
            fvc_min_threshold=0.05     # Odrzucenie FVC <= 5%
        )
        all_dfs.append(df_pair)
    else:
        print(f"Pominięto (brak pliku na dysku): {item['flight_date']} ({item['field_id']})")
        # Opcjonalnie: wydrukuj, którego pliku brakuje, żeby było łatwiej zdebugować
        if not os.path.exists(item["uav_tif"]): print(f" -> Brak UAV: {item['uav_tif']}")
        if not os.path.exists(item["ps_tif"]): print(f" -> Brak PlanetScope: {item['ps_tif']}")
        if not os.path.exists(item["geojson_path"]): print(f" -> Brak GeoJSON: {item['geojson_path']}")

# Złączenie wszystkich nalotów w jedną dużą tabelę (~22 tys. wierszy) i zapis do CSV:
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
    output_csv = "baza_kalibracyjna_UAV_PlanetScope_FVC.csv"
    final_df.to_csv(output_csv, index=False)
    print(f"\n==========================================")
    print(f"GOTOWE! Zapisano plik: {output_csv}")
    print(f"Łączna liczba oczek 3x3 m w bazie: {len(final_df)}")
    display(final_df.head())


Przetwarzanie: Pole 01W | Dron: 2026-08-10 vs Satelita: 2026-08-09
 -> Docinanie UAV do GeoJSON (01W) i liczenie wskaźników na GPU...
 -> Gotowe w 262.15 s | Wyciągnięto 3685 oczek 3x3 m.

Przetwarzanie: Pole 01W | Dron: 2026-09-02 vs Satelita: 2026-09-02
 -> Docinanie UAV do GeoJSON (01W) i liczenie wskaźników na GPU...
 -> Gotowe w 260.41 s | Wyciągnięto 2859 oczek 3x3 m.

Przetwarzanie: Pole 01W | Dron: 2026-09-09 vs Satelita: 2026-09-09
 -> Docinanie UAV do GeoJSON (01W) i liczenie wskaźników na GPU...
 -> Gotowe w 213.84 s | Wyciągnięto 2422 oczek 3x3 m.

Przetwarzanie: Pole 01W | Dron: 2026-09-15 vs Satelita: 2026-09-16
 -> Docinanie UAV do GeoJSON (01W) i liczenie wskaźników na GPU...
 -> Gotowe w 273.06 s | Wyciągnięto 2715 oczek 3x3 m.

Przetwarzanie: Pole 01W | Dron: 2026-10-01 vs Satelita: 2026-10-02
 -> Docinanie UAV do GeoJSON (01W) i liczenie wskaźników na GPU...
 -> Gotowe w 283.42 s | Wyciągnięto 2705 oczek 3x3 m.

Przetwarzanie: Pole 01W | Dron: 2026-10-03 vs Satelita

,field_id,flight_date,ps_date,row,col,x_center,y_center,FVC,n_uav_veg_pixels,UAV_NDVI,...,UAV_CIre,UAV_ARI,UAV_PRI,PS_NDVI,PS_SAVI,PS_EVI,PS_RENDVI,PS_CIre,PS_NDWI,PS_PRI
0,01W,2026-08-10,2026-08-09,3,41,427375.5,5552347.5,1.0000,7887,0.608981,...,0.475192,-1.112244,-0.117739,0.816772,0.606454,0.692752,0.536497,2.314966,-0.712967,-0.055163
1,01W,2026-08-10,2026-08-09,4,40,427372.5,5552344.5,1.0000,8246,0.589623,...,0.370735,-0.906089,-0.117963,0.805182,0.602615,0.678158,0.523132,2.194030,-0.702595,-0.028834
2,01W,2026-08-10,2026-08-09,4,41,427375.5,5552344.5,1.0000,8277,0.561736,...,0.361686,-0.779112,-0.120691,0.813764,0.603971,0.688517,0.527429,2.232168,-0.711368,-0.035961
3,01W,2026-08-10,2026-08-09,4,42,427378.5,5552344.5,1.0000,7911,0.544963,...,0.336838,-0.686279,-0.120978,0.810945,0.597313,0.673716,0.533439,2.286681,-0.709589,-0.027701
4,01W,2026-08-10,2026-08-09,5,38,427366.5,5552341.5,0.9986,4370,0.608331,...,0.365286,-0.961454,-0.111944,0.809524,0.614579,0.704771,0.515776,2.130317,-0.687352,-0.032452
